# ?????????DCT + DWT?

**??**?????  
**??**???  
**????**?2 ??  
**????**?Python ???NumPy????????DCT/DWT ??

---

## ????
- ?? DCT ? DWT ???????????
- ??????????????????
- ?? DCT ????/??? DWT ????/???????
- ???????????????????????


## ????

?????????????
1. ??????????
2. ?? DCT ??????????-??-????-?????
3. ?? DWT ????????????-????-????
4. ????????? PSNR ???????

???? JupyterLite ???????????????????? Markdown ?????????????? Python ?????


---
## Step 1??????????

?????????? Python ??
- `numpy`??????????
- `scipy.fftpack`??? DCT / IDCT
- `pywt`???????
- `PIL` / `Pillow`??????????

???????
```bash
pip install numpy scipy pywavelets pillow
```

?? `pywavelets` ?????????????????? Python ?????


In [ ]:
# ?????????????????????????
modules = {
    'numpy': 'import numpy as np',
    'scipy.fftpack': 'from scipy import fftpack',
    'pywt': 'import pywt',
    'PIL': 'from PIL import Image'
}

for name, stmt in modules.items():
    try:
        exec(stmt, {})
        print(f'[OK] {name} ??')
    except Exception as exc:
        print(f'[WARN] {name} ???: {exc}')


---
## Step 2?DCT ??????

DCT ????????
1. ????????? `8x8`??
2. ??????? DCT?
3. ?????????????
4. ?? IDCT ?????

?????????????
- ????????????????
- ???????????????????


In [ ]:
import numpy as np
from scipy import fftpack

class DCTWatermark:
    def __init__(self, alpha=0.2, block_size=8):
        self.alpha = alpha
        self.block_size = block_size

    def _dct2(self, block):
        return fftpack.dct(fftpack.dct(block.T, norm='ortho').T, norm='ortho')

    def _idct2(self, block):
        return fftpack.idct(fftpack.idct(block.T, norm='ortho').T, norm='ortho')

# ?? DCT / IDCT ????
dct_wm = DCTWatermark(alpha=0.2, block_size=8)
test_block = np.random.randint(0, 256, (8, 8)).astype(np.float64)
dct_block = dct_wm._dct2(test_block)
recovered_block = dct_wm._idct2(dct_block)
mse = np.mean((test_block - recovered_block) ** 2)

print(f'8x8 ? DCT-IDCT ??? MSE: {mse:.8f}')
print('???? 4x4:
', test_block[:4, :4].round(2))
print('DCT ??? 4x4:
', dct_block[:4, :4].round(2))
print('???? 4x4:
', recovered_block[:4, :4].round(2))


### DCT ???????

???????
- ???? `256x256` ??????
- ?? `32x32` ?????
- ???????????????
- ???? PSNR ???????


In [ ]:
import numpy as np

class DCTWatermark:
    def __init__(self, alpha=0.3, block_size=8):
        self.alpha = alpha
        self.block_size = block_size

    def _dct2(self, block):
        from scipy import fftpack
        return fftpack.dct(fftpack.dct(block.T, norm='ortho').T, norm='ortho')

    def _idct2(self, block):
        from scipy import fftpack
        return fftpack.idct(fftpack.idct(block.T, norm='ortho').T, norm='ortho')

    def embed_watermark(self, image, watermark):
        image = image.astype(np.float64).copy()
        h, w = image.shape
        wm_h, wm_w = watermark.shape
        blocks_h = h // self.block_size
        blocks_w = w // self.block_size
        resized = watermark[:blocks_h, :blocks_w]
        out = image.copy()

        for i in range(resized.shape[0]):
            for j in range(resized.shape[1]):
                r0 = i * self.block_size
                c0 = j * self.block_size
                block = out[r0:r0+self.block_size, c0:c0+self.block_size]
                coeff = self._dct2(block)
                coeff[3, 4] += self.alpha * (1 if resized[i, j] > 0 else -1) * max(abs(coeff[3, 4]), 1.0)
                out[r0:r0+self.block_size, c0:c0+self.block_size] = self._idct2(coeff)

        return np.clip(out, 0, 255).astype(np.uint8)

    def extract_watermark(self, image, shape):
        h, w = image.shape
        blocks_h = h // self.block_size
        blocks_w = w // self.block_size
        out = np.zeros((blocks_h, blocks_w), dtype=int)

        for i in range(blocks_h):
            for j in range(blocks_w):
                r0 = i * self.block_size
                c0 = j * self.block_size
                block = image[r0:r0+self.block_size, c0:c0+self.block_size].astype(np.float64)
                coeff = self._dct2(block)
                out[i, j] = 1 if coeff[3, 4] > 0 else 0

        return out[:shape[0], :shape[1]]

image = np.random.randint(100, 200, (256, 256), dtype=np.uint8)
watermark = np.zeros((32, 32), dtype=int)
watermark[8:24, 8:24] = 1

model = DCTWatermark(alpha=0.3, block_size=8)
watermarked = model.embed_watermark(image, watermark)
extracted = model.extract_watermark(watermarked, watermark.shape)

mse = np.mean((image.astype(float) - watermarked.astype(float)) ** 2)
psnr = float('inf') if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))
similarity = np.mean(watermark == extracted)

print(f'DCT ?? PSNR: {psnr:.2f} dB')
print(f'DCT ?????: {similarity * 100:.2f}%')


---
## Step 3?DWT ??????

DWT??????????????????????? `cA`?
- ???????????????
- ?????????????

?????
1. ???????????
2. ?????? `cA`?
3. ?????????? `cA`?
4. ?????
5. ?????????????????????


In [ ]:
import numpy as np
import pywt

class DWTWatermark:
    def __init__(self, alpha=0.25, wavelet='haar', level=2):
        self.alpha = alpha
        self.wavelet = wavelet
        self.level = level

    def embed_watermark(self, image, watermark):
        coeffs = pywt.wavedec2(image.astype(np.float64), wavelet=self.wavelet, level=self.level)
        cA = coeffs[0]
        wm = np.resize(watermark.astype(np.float64), cA.shape)
        cA_wm = cA + self.alpha * wm * np.maximum(np.abs(cA), 1.0)
        new_coeffs = [cA_wm] + list(coeffs[1:])
        out = pywt.waverec2(new_coeffs, wavelet=self.wavelet)
        out = out[:image.shape[0], :image.shape[1]]
        return np.clip(out, 0, 255).astype(np.uint8)

    def extract_watermark(self, watermarked_image, original_image, shape):
        coeffs_orig = pywt.wavedec2(original_image.astype(np.float64), wavelet=self.wavelet, level=self.level)
        coeffs_wm = pywt.wavedec2(watermarked_image.astype(np.float64), wavelet=self.wavelet, level=self.level)
        cA_orig = coeffs_orig[0]
        cA_wm = coeffs_wm[0]
        recovered = (cA_wm - cA_orig) / (self.alpha * np.maximum(np.abs(cA_orig), 1.0))
        recovered = (recovered > 0).astype(int)
        return recovered[:shape[0], :shape[1]]

image = np.random.randint(100, 200, (256, 256), dtype=np.uint8)
watermark = np.zeros((32, 32), dtype=int)
watermark[12:20, 12:20] = 1

model = DWTWatermark(alpha=0.25, wavelet='haar', level=2)
watermarked = model.embed_watermark(image, watermark)
extracted = model.extract_watermark(watermarked, image, watermark.shape)

mse = np.mean((image.astype(float) - watermarked.astype(float)) ** 2)
psnr = float('inf') if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))
similarity = np.mean(watermark == extracted)

print(f'DWT ?? PSNR: {psnr:.2f} dB')
print(f'DWT ?????: {similarity * 100:.2f}%')


---
## Step 4?DCT ? DWT ????

????????????????

1. **????**
   - DCT???????
   - DWT?????????

2. **????**
   - ??????DCT ????????? PSNR
   - DWT ??????????????

3. **???**
   - DCT ??????? JPEG ??
   - DWT ????????????????

4. **?????**
   - DCT??????????
   - DWT????????????????

????? `alpha`????????????????? PSNR ????????
